# 02 – Preprocessing
Ziel: 10-Min → stündlich, Features bauen, final_dataset.csv speichern.

In [1]:
import importlib
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

import src.data.load_data as load_data_module
import src.data.feature_engineering as feature_engineering_module

importlib.reload(load_data_module)
importlib.reload(feature_engineering_module)

load_production = load_data_module.load_production
get_production_path = load_data_module.get_production_path
build_final_dataset = feature_engineering_module.build_final_dataset

## Rohdaten laden

In [2]:
print(f'Verwende Produktionsdatei: {get_production_path()}')
df_raw = load_production()
print(f'Rohdaten: {df_raw.shape} Zeilen')
df_raw.head()

Verwende Produktionsdatei: C:\Users\felix\Documents\AI1\wind_bidding_project\data\raw\Daten zur Windkennlinie_2025-09-17\Schwanfeld\2016_10-Min-Daten WEA Schwanfeld.xlsx
Rohdaten: (39359, 17) Zeilen


,Seriennummer,Datum,Zählerstand,Betriebsstunden,Windgeschwindigkeit (min) [m/s],Windgeschwindigkeit (Ø) [m/s],Windgeschwindigkeit (max) [m/s],Rotordrehzahl (min) [U/min],Rotordrehzahl (Ø) [U/min],Rotordrehzahl (max) [U/min],Blindleistung (min) [kW],Blindleistung (Ø) [kW],Blindleistung (max) [kW],Leistung (min) [kW],Leistung (Ø) [kW],Leistung (max) [kW],Gondelposition [°]
0,1150229,31.03.2016 11:37,204.0,0.0,0.1,2.5,4.2,0.0,0,0.04,-13.0,-3,0.0,-44.0,-16,0.0,64.0
1,1150229,31.03.2016 11:40,204.0,0.0,0.5,2.8,4.5,0.0,0,0.00,-3.0,-2,-2.0,-15.0,-9,-7.0,64.0
2,1150229,31.03.2016 11:50,204.0,0.0,0.3,2.4,4.1,0.0,0,0.00,-3.0,-3,-2.0,-10.0,-8,-7.0,64.0
3,1150229,31.03.2016 12:00,204.0,0.0,0.2,2.7,4.5,0.0,0.01,0.08,-12.0,-3,-2.0,-15.0,-8,-7.0,64.0
4,1150229,31.03.2016 12:10,204.0,0.0,0.1,2.4,4.8,0.0,0,0.05,-3.0,-2,-2.0,-10.0,-8,-7.0,64.0


## Auf Stundenwerte aggregieren + Features bauen

In [3]:
df = build_final_dataset(df_raw)
print(f'Stündliche Daten: {df.shape} Zeilen')
df.head(10)

c:\Users\felix\Documents\AI1\wind_bidding_project\notebooks\..\src\data\feature_engineering.py:32: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["timestamp"] = pd.to_datetime(df[TIMESTAMP_COL])


Stündliche Daten: (6600, 10) Zeilen


,timestamp,power,wind_speed,hour,dayofweek,month,hour_sin,hour_cos,dow_sin,dow_cos
0,2016-03-31 11:00:00,-11.0,2.566667,11,3,3,2.588190e-01,-9.659258e-01,0.433884,-0.900969
1,2016-03-31 12:00:00,-7.666667,3.616667,12,3,3,1.224647e-16,-1.000000e+00,0.433884,-0.900969
2,2016-03-31 13:00:00,30.666667,3.533333,13,3,3,-2.588190e-01,-9.659258e-01,0.433884,-0.900969
3,2016-03-31 14:00:00,169.833333,4.783333,14,3,3,-5.000000e-01,-8.660254e-01,0.433884,-0.900969
4,2016-03-31 15:00:00,45.0,0.8,15,3,3,-7.071068e-01,-7.071068e-01,0.433884,-0.900969
5,2016-03-31 16:00:00,0.0,0.0,16,3,3,-8.660254e-01,-5.000000e-01,0.433884,-0.900969
6,2016-03-31 17:00:00,0.0,0.0,17,3,3,-9.659258e-01,-2.588190e-01,0.433884,-0.900969
7,2016-03-31 18:00:00,0.0,0.0,18,3,3,-1.000000e+00,-1.836970e-16,0.433884,-0.900969
8,2016-03-31 19:00:00,0.0,0.0,19,3,3,-9.659258e-01,2.588190e-01,0.433884,-0.900969
9,2016-03-31 20:00:00,0.0,0.0,20,3,3,-8.660254e-01,5.000000e-01,0.433884,-0.900969


## Qualitätsprüfung

In [4]:
print('Fehlende Werte:')
print(df.isnull().sum())
print(f'\nZeitraum: {df.timestamp.min()} bis {df.timestamp.max()}')
print(f'Anzahl Stunden: {len(df)}')
print(f'Vollständig? {len(df)} / {365*24} erwartet')

Fehlende Werte:
timestamp     0
power         0
wind_speed    0
hour          0
dayofweek     0
month         0
hour_sin      0
hour_cos      0
dow_sin       0
dow_cos       0
dtype: int64

Zeitraum: 2016-03-31 11:00:00 bis 2016-12-31 23:00:00
Anzahl Stunden: 6600
Vollständig? 6600 / 8760 erwartet


## Datensatz speichern
Dieser CSV ist der einzige Input für alle weiteren Notebooks.

In [5]:
df.to_csv('../data/processed/final_dataset.csv', index=False)
print('Gespeichert: data/processed/final_dataset.csv')
df.describe()

Gespeichert: data/processed/final_dataset.csv


,timestamp,hour,dayofweek,month,hour_sin,hour_cos,dow_sin,dow_cos
count,6600,6600.000000,6600.000000,6600.000000,6600.000000,6.600000e+03,6600.000000,6600.000000
mean,2016-08-16 04:10:43.090909,11.527727,3.009091,7.996061,-0.002150,-1.493872e-03,-0.003854,-0.006113
min,2016-03-31 11:00:00,0.000000,0.000000,3.000000,-1.000000,-1.000000e+00,-0.974928,-0.900969
25%,2016-06-08 04:45:00,6.000000,1.000000,6.000000,-0.707107,-7.071068e-01,-0.781831,-0.900969
50%,2016-08-16 03:30:00,12.000000,3.000000,8.000000,0.000000,-1.836970e-16,0.000000,-0.222521
75%,2016-10-24 02:15:00,18.000000,5.000000,10.000000,0.707107,7.071068e-01,0.781831,0.623490
max,2016-12-31 23:00:00,23.000000,6.000000,12.000000,1.000000,1.000000e+00,0.974928,1.000000
std,NaN,6.917597,1.995048,2.588043,0.707296,7.070196e-01,0.706863,0.707421


## Preisdaten
DA- und reBAP-Preise werden **nicht** in diesen Datensatz gemergt.
Sie werden direkt in  geladen,
da sie nur für die wirtschaftliche Evaluation benötigt werden und nicht direkt für den Forecast.